# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )     
        
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
nvidia/LocateAnything-3B
Updated
7 days ago
•
78.9k
•
1.13k
LiquidAI/LFM2.5-8B-A1B
Updated
about 1 hour ago
•
60.2k
•
472
openbmb/MiniCPM5-1B
Updated
9 days ago
•
68.5k
•
754
HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
Apr 17
•
2.6M
•
1.33k
stepfun-ai/Step-3.7-Flash
Updated


In [21]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
 You are an assistant that analyzes the contents of several relevant pages from a company website
 and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
 Respond in markdown without code blocks.
 Include details of company culture, customers and careers/jobs if you have the information.
 """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nnvidia/LocateAnything-3B\nUpdated\n7 days ago\n•\n78.9k\n•\n1.13k\nLiquidAI/LFM2.5-8B-A1B\nUpdated\nabout 1 hour ago\n•\n60.2k

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face - The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a leading collaboration platform for the machine learning community, serving as the central hub where millions of AI practitioners, researchers, and developers create, discover, and share state-of-the-art models, datasets, and applications. The company prides itself on enabling collaboration at unprecedented scale — supporting over 2 million machine learning models and 500,000 datasets and hosting more than 1 million applications created by its vibrant community.

---

## What We Offer

- **Models:** Access and contribute to a vast repository of 2M+ cutting-edge machine learning models covering a wide variety of AI tasks.
  
- **Datasets:** Browse and utilize over 500k+ public datasets supporting research and model training with easy collaboration tools.
  
- **Spaces:** Deploy and share machine learning demos and apps in an intuitive, ready-to-use environment.
  
- **Buckets:** Secure cloud storage solutions tailored for AI data and model needs.
  
- **HuggingChat:** An advanced conversational AI platform.
  
- **Enterprise Solutions:** Offering Hugging Face PRO, inference providers, endpoints, and enterprise support to accelerate AI deployment.

---

## Company Culture & Community

Hugging Face embodies an open, inclusive, and collaborative culture that encourages innovation through community-driven efforts. The platform nurtures a global AI community spanning researchers, engineers, students, and firms who actively share knowledge via forums, Discord channels, blogs, and daily AI papers. The ecosystem fosters a culture of learning, transparency, and mutual support, making it a premier destination for machine learning collaboration.

With initiatives like hackathons and community spaces, Hugging Face encourages hands-on contributions, experimentation, and the democratization of AI technology worldwide.

---

## Our Customers & Partners

Hugging Face serves a diverse range of customers:

- Independent AI researchers and developers looking to access high-quality models and datasets.
- Startups and enterprises seeking scalable AI solutions and support for deploying state-of-the-art machine learning applications.
- Educational institutions and students leveraging the platform for research and learning.
- Industry leaders in technology, healthcare, finance, and more integrating Hugging Face tools for innovation.

Leading organizations and projects often utilize Hugging Face as their machine learning collaboration backbone.

---

## Careers at Hugging Face

Join the forefront of AI innovation! Hugging Face offers exciting career opportunities for passionate individuals eager to:

- Contribute to open-source AI projects.
- Develop scalable cloud AI services.
- Collaborate with a global, diverse team of experts.
- Impact the future of machine learning and AI applications.

The company fosters a culture of inclusion, continuous learning, and creativity. Candidates can expect to engage in challenging technical problems while being supported by a collaborative community-driven environment.

---

## Get Involved

- Explore AI applications and models at [huggingface.co](https://huggingface.co).
- Join community discussions on Discord and forums.
- Contribute to open-source projects on GitHub.
- Attend hackathons and workshops hosted regularly.
- Discover enterprise solutions tailored for your organization's AI needs.

---

### Brand Identity

Hugging Face's vibrant brand reflects its energetic and forward-thinking ethos, featuring the signature colors #FFD21E (yellow), #FF9D00 (orange), and #6B7280 (gray). The recognizable logo and assets are freely available for use in projects aligned with the community spirit.

---

**Hugging Face**  
*The AI community building the future*  
Create, discover, and collaborate on machine learning better – together.

---

For more information, visit [huggingface.co](https://huggingface.co) and join the AI revolution today!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


# Hugging Face - The AI Community Building the Future

---

## About Hugging Face

Hugging Face is the leading collaboration platform at the heart of the AI revolution. It serves as a central hub where the machine learning community — including engineers, scientists, and end users — can share, explore, and experiment with open-source machine learning (ML) models, datasets, and applications. With over 2 million models and 500,000 datasets available, Hugging Face empowers the next generation of AI innovators to build an open, ethical, and collaborative AI future.

---

## What Hugging Face Offers

- **Models:** Access and contribute to a vast library of over 2 million open-source ML models covering a wide range of AI tasks.
  
- **Datasets:** Browse and share over 500,000 datasets that support training and benchmarking AI systems.

- **Spaces:** Host and explore AI apps and demos created by the community, showcasing cutting-edge ML applications.

- **Enterprise Solutions:** Tailored team and enterprise plans including:
  - Hugging Face PRO for enhanced collaboration
  - Enterprise Support for production environments
  - Inference Providers and Endpoints for scalable model deployment
  - Storage Buckets for managing data and assets securely

- **Community & Learning:** Join an active ecosystem with forums, Discord channels, GitHub repositories, and a rich blog that includes daily research papers and AI insights.

---

## Company Culture & Mission

Hugging Face fosters a vibrant, open, and inclusive community culture. It is dedicated to transparency, ethical AI development, and democratizing machine learning technology. The company supports continuous learning and collaboration to ensure that AI benefits everyone equally.

- Encourages open-source contributions
- Supports diversity and inclusion in AI research
- Provides learning resources and community engagement platforms

---

## Customers and Collaborators

Hugging Face serves a diverse ecosystem of customers including:

- Independent ML developers and researchers worldwide
- Tech companies looking to integrate AI models and infrastructure
- Enterprises requiring robust ML deployment and support services
- Academic and research institutions advancing AI studies

---

## Careers & Opportunities

Joining Hugging Face means becoming part of a fast-growing team dedicated to moving the AI industry forward through innovation and openness. The company offers:

- Roles across engineering, science, product, and community management
- An environment driven by passion for open-source and ethical AI
- Opportunities to work with a talented team pushing the boundaries of machine learning technology

Explore careers via the Hugging Face website and contribute to shaping the future of AI.

---

## Brand and Visuals

Hugging Face's brand uses vibrant colors: 

- Yellow (#FFD21E) 
- Orange (#FF9D00)
- Gray (#6B7280)

Official logo and brand assets are available for use in projects promoting open-source and AI collaboration.

---

## Join the Future with Hugging Face

- Collaborate on the world's largest and fastest-growing ML platform
- Access cutting-edge models, datasets, and AI apps
- Be part of a mission-driven community building responsible AI technology

Visit [https://huggingface.co](https://huggingface.co) to explore, collaborate, and innovate.

---

*Hugging Face - Advancing AI together.*

In [22]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Welcome to Hugging Face  
*The AI community building the future (with hugs and code)* 🤗💻

---

## Who Are We?  
Hugging Face is not just a company – it's a bustling AI playground and collaboration platform where the smartest machine learning people (and bots) come together to create the future of artificial intelligence. Think of us as the friendly neighborhood hub where 2 million+ models, datasets, and AI applications hang out, share ideas, and push ML boundaries.

Whether you’re an ML engineer, data scientist, researcher, or just plain curious, Hugging Face is your go-to place to explore, experiment, and innovate.

---

## What’s Inside the Hugging Face Universe?  

- **Models & Datasets Galore:** Explore over 2 million ready-to-rock models and half a million datasets curated by a passionate community.
- **Spaces:** Launch and share your creative AI apps with zero hassle. From image recognition to talking-head video generation, there's always something amazing running in Spaces.
- **HuggingChat:** Your friendly AI chat companion right on the platform.
- **Enterprise Solutions:** Power your business with PRO services, scalable inference endpoints, and expert support.
- **Community & Collaboration:** Join lively discussions on Discord, GitHub, and our forum where collaboration is king.

---

## Our Culture  
At Hugging Face, we believe in:

- **Open & Ethical AI:** Creating transparent AI that everyone can inspect, learn from, and improve.
- **Collaboration Over Competition:** Together, we build better models and share knowledge.
- **Playfulness:** We hug (metaphorically!) problems with a smile and a bit of wit.
- **Continuous Learning:** Our team and community thrive on curiosity and sharing discoveries.
- **Innovation at the Edge:** From daily papers to cutting-edge experiments, we live on the frontier of AI tech.

(And yes, we love a good meme in our Discord channels.)

---

## Who Uses Hugging Face?  

- **Researchers & Academics:** Access datasets, publish models, and push scientific boundaries.
- **Startups & Enterprises:** Build AI-powered products without reinventing the wheel.
- **Developers Everywhere:** From hobbyists to pros, create, share, and level up your AI game.
- **AI Enthusiasts:** Join a vibrant, global community that loves pushing ML forward.

---

## Careers – Join the Hug Squad!  

Are you:

- Passionate about AI and open-source?
- Ready to work with the sharpest minds on the most exciting ML projects?
- Eager to impact millions through accessible AI tools and community building?

If yes, Hugging Face offers you **a playground with purpose**, competitive perks, and a culture that’s more about hugs than hustle (but we hustle well too!).

Check out our open roles, toss your hat in the ring, and help us build the future of AI — one model at a time.  

---

## Quick Facts to Impress Your Friends  

- 2 million+ models available  
- Half a million+ datasets to explore  
- Zero friction deployment with Spaces  
- Community growing faster than AI hype cycles  
- Official brand colors: sunny #FFD21E and fiery #FF9D00 (because science is bright and spicy)

Want to dive in? Visit [huggingface.co](https://huggingface.co) where the future of AI is just a click away.

---

*Hugging Face* — building open, ethical AI, with a bit of love, a lot of code, and endless community spirit. Come join the party! 🎉🤗

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>